# RAG lecture harness

Runs every code cell of `RAG_Lecture.ipynb` (branch `colab/wip`) in one kernel, prints a per-cell verdict, and stops at the first failure with its traceback.

In [ ]:

# Runs every cell of RAG_Lecture.ipynb in order and prints what breaks.
import json, subprocess, traceback, urllib.request
import nbformat

BRANCH = "colab/wip"
URL = f"https://raw.githubusercontent.com/LorMolf/knowledge-enhanced-llm-lectures/{BRANCH}/rag_lecture/RAG_Lecture.ipynb"

print("fetching", URL)
raw = urllib.request.urlopen(URL, timeout=120).read().decode()
nb = nbformat.reads(raw, as_version=4)
code_cells = [c for c in nb.cells if c.cell_type == "code"]
print("cells:", len(nb.cells), "| code:", len(code_cells))

ns = {"__name__": "__main__"}
report = []
for i, cell in enumerate(code_cells, 1):
    src = cell.source
    head = src.strip().splitlines()[0][:60] if src.strip() else "(empty)"
    if src.lstrip().startswith("%%"):
        cmd = src.split("\n", 1)[1]
        done = subprocess.run(["bash", "-lc", cmd], capture_output=True, text=True)
        tail = (done.stdout + done.stderr).strip().splitlines()[-3:]
        status = "OK" if done.returncode == 0 else "ERR"
        print(f"CELL {i:03} {status} shell :: {' | '.join(tail)[:200]}", flush=True)
        report.append({"cell": i, "status": status, "kind": "shell", "head": head, "rc": done.returncode})
        if status == "ERR":
            break
        continue
    try:
        exec(compile(src, f"cell{i}", "exec"), ns)
        print(f"CELL {i:03} OK   {head}", flush=True)
        report.append({"cell": i, "status": "OK", "kind": "python", "head": head})
    except Exception:
        tb = traceback.format_exc()
        print(f"CELL {i:03} ERR  {head}", flush=True)
        print(tb[-1500:], flush=True)
        report.append({"cell": i, "status": "ERR", "kind": "python", "head": head, "traceback": tb[-1500:]})
        break

bad = [r for r in report if r["status"] == "ERR"]
print("\n==== SUMMARY ====")
print(f"ran {len(report)} of {len(code_cells)} | failures: {len(bad)}")
for r in bad:
    print("FAIL cell", r["cell"], "::", r["head"])
json.dump(report, open("/content/report.json", "w"), indent=1)
print("report saved to /content/report.json")
